# RFSoC Sounder Class Demo

This notebook uses `RFSoCSounder` to:

1. Configure clocks and load the overlay.
2. Run startup self-checks, including an ADC capture smoke test.
3. Configure the DAC/ADC Fine NCO at 1 GHz.
4. Generate and load a 1024-point band-limited Zadoff–Chu waveform.
5. Transmit for one second and capture one RX frame.
6. Plot noise-referenced spectra and channel magnitude/phase.

Use the same verified DAC-to-ADC loopback path with appropriate attenuation.


In [ ]:
from pathlib import Path
import sys

PROJECT_DIR = Path.cwd()
MODULE_DIR = PROJECT_DIR

if str(MODULE_DIR) not in sys.path:
    sys.path.insert(0, str(MODULE_DIR))

from rfsoc_sounder import RFSoCSounder


In [ ]:
sounder = RFSoCSounder(
    project_dir=PROJECT_DIR,
    bit_name="rfsoc_waveform_tx_rx.bit",
)

startup_report = sounder.startup(
    configure_clocks=True,
    lmk_freq_mhz=245.76,
    lmx_freq_mhz=491.52,
    download=True,
    run_self_test=True,
    capture_smoke_test=True,
)

sounder.print_self_test(startup_report)
print()
print(sounder.summary())


In [ ]:
tx_settings = sounder.configure_tx(
    center_mhz=1000.0,
    nyquist_zone=1,
    duration_s=0.05,
    phase_offset_deg=0.0,
)

rx_settings = sounder.configure_rx(
    center_mhz=1000.0,
    nyquist_zone=1,
    calibration_mode=2,
    dither=True,
    capture_timeout_s=0.5,
    settle_s=0.020,
)

print("TX settings:", tx_settings)
print("RX settings:", rx_settings)


In [ ]:
waveform = sounder.set_zadoff_chu(
    sc_range=(-250, 250),
    root=3,
    zc_q=0.0,
    amplitude=0.9,
)

load_report = sounder.load_waveform(verify=True)

print("Waveform I range:", int(waveform.i.min()), "to", int(waveform.i.max()))
print("Waveform Q range:", int(waveform.q.min()), "to", int(waveform.q.max()))
print("Load report:", load_report)


In [ ]:
capture = sounder.transmit_and_capture(
    duration_s=0.040,
    settle_s=0.01,
    capture_timeout_s=0.5,
    auto_load_waveform=True,
    require_frame_count_match=True,
)

print("Captured words:", capture.captured_words)
print("Frames requested:", capture.frames_requested)
print("Frames completed:", capture.frames_completed)
print("Capture time:", capture.capture_time_s, "s")
print("Total time:", capture.total_time_s, "s")


In [ ]:
stats = sounder.capture_statistics(capture)

for key, value in stats.items():
    print(f"{key:24s}: {value}")


In [ ]:
spectrum = sounder.plot_spectra(
    capture,
    rf_axis=True,
    noise_referenced=True,
    floor_db=-30.0,
)

print("TX mean active/noise:",
      spectrum["tx_mean_active_above_noise_db"], "dB")
print("RX mean active/noise:",
      spectrum["rx_mean_active_above_noise_db"], "dB")


In [ ]:
channel_standard = sounder.plot_channel(
    capture,
    definition="standard",
    rf_axis=True,
)

print(
    "Estimated standard-channel delay:",
    channel_standard["estimated_delay_s"] * 1e9,
    "ns",
)


In [ ]:
# Optional: use the earlier requested cross-spectrum convention
# H_cross(f) = S(f) * conj(R(f)).
channel_cross = sounder.plot_channel(
    capture,
    definition="cross",
    rf_axis=True,
)

print(
    "Estimated cross-spectrum delay:",
    channel_cross["estimated_delay_s"] * 1e9,
    "ns",
)
